In [12]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon
import h3

In [13]:
gdf = gpd.read_file('../data/geography-levels/zonas-montevideo-actualizadas-steer.gpkg')
resolution=7

In [14]:
def tessellate(gdf, resolution):
    gdf = gdf.to_crs("EPSG:4326")

    # Use overlap containment so hexes covering small/thin zones aren't dropped
    # (default center-based mode misses any polygon smaller than an h3 cell).
    hex_ids = set()
    uncovered = []
    for idx, geometry in gdf.geometry.items():
        shape = h3.geo_to_h3shape(geometry)
        cells = h3.h3shape_to_cells_experimental(shape, resolution, contain="overlap")
        if not cells:
            uncovered.append(idx)
        hex_ids.update(cells)

    print(f"zones with no covering hex: {len(uncovered)} / {len(gdf)}")

    # `cell_to_boundary` returns (lat, lng) tuples; shapely expects (lng, lat).
    gdf_new = gpd.GeoDataFrame(
        {"hex_id": list(hex_ids)},
        geometry=[
            Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(h)])# return gdf
            for h in hex_ids
        ],)
    gdf_new.set_crs('EPSG:4326', inplace=True)
    return gdf_new

In [16]:
gdf_new = tessellate(gdf, 7)
gdf_new.to_file("../output/AMM_h3_level{}.gpkg".format(resolution))

zones with no covering hex: 0 / 630
